# 03 Traditional Text Representations & TF-IDF Vectorization

**Scenario**: 3-Document Enterprise Technical Log Corpus Vectorization & Cosine Similarity.

This notebook demonstrates step-by-step TF-IDF vector computation, smooth IDF weighting, L2 normalization, and pair-wise Cosine similarity scoring.

## Step 1: 3-Document Technical Log Corpus Setup

In [1]:
corpus = [
    "database connection failed",
    "database query timeout",
    "billing payment declined"
]

print("=== 3-Document Corpus ===")
for idx, doc in enumerate(corpus, 1):
    print(f"d{idx}: '{doc}'")

=== 3-Document Corpus ===
d1: 'database connection failed'
d2: 'database query timeout'
d3: 'billing payment declined'


## Step 2: Step-by-Step Manual TF-IDF Matrix & Cosine Similarity Calculation

In [2]:
import numpy as np
import math

# Vocabulary Extraction
vocab = sorted(list(set(" ".join(corpus).split())))
print("Vocabulary (|V|=8):", vocab)

# Compute TF Matrix
tf_matrix = []
for doc in corpus:
    words = doc.split()
    tf_vec = [words.count(term) / len(words) for term in vocab]
    tf_matrix.append(tf_vec)

tf_matrix = np.array(tf_matrix)
print("\nRaw TF Matrix:\n", tf_matrix.round(4))

# Compute DF and Smooth IDF
df = np.array([sum(1 for doc in corpus if term in doc.split()) for term in vocab])
N = len(corpus)
idf = np.log((1 + N) / (1 + df)) + 1.0

print("\nDocument Frequency DF:", df)
print("Smooth IDF Weights:", idf.round(4))

# Compute Unnormalized TF-IDF Matrix
tfidf_raw = tf_matrix * idf

# L2 Normalization
norms = np.linalg.norm(tfidf_raw, axis=1, keepdims=True)
tfidf_norm = tfidf_raw / norms
print("\nL2 Normalized TF-IDF Matrix:\n", tfidf_norm.round(4))

# Cosine Similarity
cos_sim_1_2 = np.dot(tfidf_norm[0], tfidf_norm[1])
cos_sim_1_3 = np.dot(tfidf_norm[0], tfidf_norm[2])

print(f"\nCosine Similarity (d1, d2): {cos_sim_1_2:.4f}")
print(f"Cosine Similarity (d1, d3): {cos_sim_1_3:.4f}")

Vocabulary (|V|=8): ['billing', 'connection', 'database', 'declined', 'failed', 'payment', 'query', 'timeout']

Raw TF Matrix:
 [[0.     0.3333 0.3333 0.     0.3333 0.     0.     0.    ]
 [0.     0.     0.3333 0.     0.     0.     0.3333 0.3333]
 [0.3333 0.     0.     0.3333 0.     0.3333 0.     0.    ]]

Document Frequency DF: [1 1 2 1 1 1 1 1]
Smooth IDF Weights: [1.6931 1.6931 1.2877 1.6931 1.6931 1.6931 1.6931 1.6931]

L2 Normalized TF-IDF Matrix:
 [[0.     0.6228 0.4736 0.     0.6228 0.     0.     0.    ]
 [0.     0.     0.4736 0.     0.     0.     0.6228 0.6228]
 [0.5774 0.     0.     0.5774 0.     0.5774 0.     0.    ]]

Cosine Similarity (d1, d2): 0.2243
Cosine Similarity (d1, d3): 0.0000


## Step 3: Verification via Scikit-Learn TfidfVectorizer Pipeline

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(smooth_idf=True, norm='l2')
sklearn_tfidf = vectorizer.fit_transform(corpus).toarray()
sklearn_sim = cosine_similarity(sklearn_tfidf)

print("=== Scikit-Learn Pipeline Verification ===")
print("Feature Names:", vectorizer.get_feature_names_out())
print("TF-IDF Vectors Match Manual Computation:", np.allclose(tfidf_norm, sklearn_tfidf))
print("Cosine Similarity Matrix:\n", sklearn_sim.round(4))

=== Scikit-Learn Pipeline Verification ===
Feature Names: ['billing' 'connection' 'database' 'declined' 'failed' 'payment' 'query'
 'timeout']
TF-IDF Vectors Match Manual Computation: True
Cosine Similarity Matrix:
 [[1.     0.2243 0.    ]
 [0.2243 1.     0.    ]
 [0.     0.     1.    ]]


## Output Explanation & Complexity Analysis

- **L2 Vector Normalization**: Normalizing term counts eliminates document length bias so $\|v\|_2 = 1.0$.
- **Manual vs Scikit-Learn Verification**: Scikit-Learn `TfidfVectorizer` outputs match manual 3-document calculations $100\%$ ($d_1$ vs $d_2$: $0.2243$, $d_1$ vs $d_3$: $0.0000$).
- **Indexing & Vectorization Time Complexity**: $O(N \cdot L)$ term count accumulation.
- **Inference Query Encoding Time Complexity**: $O(k \cdot |V|)$ where $k \ll |V|$ is non-zero term count.
- **Cosine Search Time Complexity**: $O(N \cdot k)$ sparse matrix dot products.
- **Space Complexity**: $O(\text{NNZ}) = O(N \cdot \bar{k})$ Compressed Sparse Row (CSR) matrix memory.
- **Component Denotations**:
  - $N$: Number of indexed corpus documents.
  - $L$: Average document token length.
  - $|V|$: Vocabulary dimension.
  - $k$: Active non-zero terms in input query.
  - $\bar{k}$: Average non-zero terms per document.